In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/img-capstone/AI_gen_img_capstone.png



## Problem : Doctor has to devote his time in writing prescription for the patient. Even the Doctor writes in hurry , where the prescription is not even readable (patient cannot get what is written on the precription).?

## Solution : Agent which writes an accurate authentic prescription for Doctor and waits for approval by Doctor and lastly prints it.This saves Doctor's time and let's it devote more time to other patients , where patients even can read the written prescription.


## Workflow:
### Stage 1:
Agent 1 : **Ayurvedic prescriber**

Agent 2 : **Homeopathic prescriber**

Agent 3 : **Allopathic prescriber**

these agents **work in parallel**

give indivisual outputs to **Aggregator Agent**

which in turn aggregates it and hands over to critique agent 

### Stage 2:
**Critique Agent** which is Looped in a **Refinement loop** where the prescription is finalized 

### Stage 3:
Prescription is presented to Doctor waiting for it's approval (**Human In the Loop** Workflow).

### Stage 4:
After it's approval by Doctor , **prescription is ready** and printed , ready to be given to patient.


### This Project includes usage of the following concepts:
#### 1. Parallel Workflows - reaserching and formulating prescription
#### 2. Loop Workflows - Refining prescription
#### 3. Long Running Operations (Human In The Loop) - Taking Doctor's approval to prescription.
#### 4. Building the Workflow - How the Agents pause - take Doctor's input - resume.

# ![Workflow of Agents](http://https://www.kaggle.com/datasets/shivapipy/img-capstone)

## ⚙️ Section 1: Setup
### **Install dependencies**
The Kaggle Notebooks environment includes a pre-installed version of the google-adk library for Python and its required dependencies, so we don't need to install additional packages in this notebook.

In [2]:
#!pip install google-adk

###  1.1 Configure your Gemini API Key.
#### 1.This notebook uses the Gemini API, which requires authentication.

#### 2.Authenticate in the notebook.

Run the cell below to complete authentication.

In [3]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

✅ Gemini API key setup complete.


### 1.2 Import ADK components

Now, import the specific components we will need from the Agent Development Kit and the Generative AI library. This keeps our code organized and ensures we have access to the necessary building blocks.

In [4]:
import uuid
from google.genai import types

from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.3 Helper functions

We'll define some helper functions. If you are running this outside the Kaggle environment, you don't need to do this.

In [5]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]['base_url']

    try:
        path_parts = baseURL.split('/')
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix

print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.4: Configure Retry Options

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [6]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

## Scection 2 : Buiding Agents 
## Our Multi-Agent

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to write prescription for all [yypes of medicinal paradigms] and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a multi-agent system. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent only does write prescription for homeopathy, another only for ayurved). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

**Architecture: Single Agent vs Multi-Agent Team**

## 2.1 Prescription Writer System
Let's build a system with these specialized agents:

**Homeopathic Agent**- Suggest Homeopathic Medicines for the Disease.

**Ayurvedic Agent** - Suggest Ayurvedic Medicines for the Disease.

**Allopathic Agent** - Suggest Allopathic Medicines for the Disease.

**Critic Agent** - Checks the Authenticity , accuracy of the prescription.

**Human In the Loop** - Doctor approves the prescrption to be printed.

## 2.2 Define agents
Now, let's build our agent. We'll configure an Agent by setting its key properties, which tell it what to do and how to operate.

To learn more, check out the documentation related to agents in ADK.

These are the main properties we'll set:


* **name** and **description**: A simple name and description to identify our agent.
* **model**: The specific LLM that will power the agent's reasoning. We'll use "gemini-2.5-flash-lite".
* **instruction**: The agent's guiding prompt. This tells the agent its goal is and how to behave.
* **tools**: A list of tools that the agent can use. To start, we'll give it the google_search tool, which lets it find up-to-date information online.
  

## 2.3 Parallel Workflows - Independent Researchers

We have several tasks that are not dependent on each other.

The Solution: Concurrent Execution

When you have independent tasks, you can run them all at the same time using a ParallelAgent. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

Use Parallel when: Tasks are independent, speed matters, and you can execute concurrently

In [7]:
# Homeopathic Agent: Its job is to use the google_search tool and present medicines.
homeopathic_agent = Agent(
    name="HomeopathicAgent",
    model = Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request medicines with proper dosage.
    instruction="""You are a specialized Homeopathic Doctor. Your only job is to use the
    google_search tool to find 2-3 relevant Homeopathic medicines with proper dosage for the given disease and present the medicines and dosage.""",
    tools=[google_search],
    output_key="homeopathic_findings", # The result of this agent will be stored in the session state with this key.
)

print("✅ homeopathic_agent created.")

✅ homeopathic_agent created.


In [8]:
# Ayurvedic Agent: Its job is to use the google_search tool and present medicines.
ayurvedic_agent = Agent(
    name="AyurvedicAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request medicines with proper dosage.
    instruction="""You are a specialized Ayurvedic Doctor. Your only job is to use the
    google_search tool to find 2-3 relevant Ayurvedic medicines with proper dosage for the given disease and present the medicines and dosage.""",
    tools=[google_search],
    output_key="ayurvedic_findings",# The result of this agent will be stored in the session state with this key.
)

print("✅ ayurvedic_agent created.")

✅ ayurvedic_agent created.


In [9]:
# Allopathic Agent: Its job is to use the google_search tool and present medicines..
allopathic_agent = Agent(
    name="AllopathicAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request medicines with proper dosage.
    instruction="""You are a specialized Allopathic Doctor. Your only job is to use the
    google_search tool to find 2-3 relevant Allopathic medicines with proper dosage for the given disease and present the medicines and dosage.""",
    tools=[google_search],
    output_key="allopathic_findings",# The result of this agent will be stored in the session state with this key.
)

print("✅ allopathic_agent created.")

✅ allopathic_agent created.


👉 Then we bring the agents together under a parallel agent, which is itself nested inside of a sequential agent.

This design ensures that the research agents run first in parallel, then once all of their research is complete, the aggregator agent brings together all of the research finding into a single prescription:

In [10]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""Combine these three medical prescriptions into a single executive prescription:

    **Homeopathic prescription:**
    {homeopathic_findings}
    
    **Ayurvedic prescription:**
    {ayurvedic_findings}
    
    **Allopathic prescription:**
    {allopathic_findings}
    
    Your prescription should contain only correct medicines with dosages for disease , and the tagline 'This is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes.'. The final prescription should be around 20 to 30 words.""",
    output_key="prescription_summary", # This will be the final output of the entire system.
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


In [11]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[ayurvedic_agent,homeopathic_agent,allopathic_agent],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
shoot_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


## 2.4 Loop Workflows - The Refinement Cycle¶
ParallelAgent produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our prescription is bad? We have no way to review it and ask for a rewrite.

The Solution: Iterative Refinement

When a task needs to be improved through cycles of feedback and revision, you can use a LoopAgent. A LoopAgent runs a set of sub-agents repeatedly until a specific condition is met or a maximum number of iterations is reached. This creates a refinement cycle, allowing the agent system to improve its own work over and over.

Use Loop when: Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

In [12]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a Medical practitioner and Doctor critic. Review the medical prescription provided below.
    Prescription: {prescription_summary}

    Evaluate the prescription's medicines and it's dosage. 
    -The medicines prescribed should be highly accurate.
    -The prescription should have dosage for each medicine.
    -Citations to be included in prescription.
    If the prescription is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique", # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


## 2.5 Setup an Exit loop
Now, we need a way for the loop to actually stop based on the critic's feedback. The LoopAgent itself doesn't automatically know that "APPROVED" means "stop."

We need an agent to give it an explicit signal to terminate the loop.

We do this in two parts:


* A simple Python function that the LoopAgent understands as an "exit" signal.
* An agent that can call that function when the right condition is met.

First, we'll define the exit_loop function:

In [13]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the prescription is finished and no more changes are needed."""
    return {"status": "approved", "message": "Prescription approved. Exiting refinement loop."}

print("✅ exit_loop function created.")

✅ exit_loop function created.


## 2.6 Define Refiner Agent
To let an agent call this Python function, we wrap it in a FunctionTool. Then, we create a RefinerAgent that has this tool.

👉 Notice its instructions: this agent is the "brain" of the loop. It reads the {critique} from the CriticAgent and decides whether to 
1. (1) call the exit_loop tool
   
                 or

2. (2) rewrite the prescription.

In [14]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a medical prescrition refiner. You have a prescription draft and critique.
    
    Prescription Draft: {prescription_summary}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the prescription draft to fully incorporate the feedback from the critique.""",
    
    output_key="prescription_summary", # It overwrites the story with the new, refined version.
    tools=[FunctionTool(exit_loop)], # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


Then we bring the agents together under a loop agent, which is itself nested inside of a sequential agent.

This design ensures that the system first produces an initial story draft, then the refinement loop runs up to the specified number of max_iterations:

In [15]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
prescription_refinement_loop = LoopAgent(
    name="PrescriptionRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2, # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: AggregatorAgent -> Refinement Loop.
froot_agent = SequentialAgent(
    name="PrescriptionPipeline",
    sub_agents=[shoot_agent , prescription_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


## 3 Long-Running Operations (Human-in-the-Loop)¶
So far, all tools execute and return immediately:

User asks → Agent calls tool → Tool returns result → Agent responds

But what if your tools are long-running or you need human approval before completing an action?

Example: A shipping agent should ask for approval before placing a large order.

User asks → Agent calls tool → Tool PAUSES and asks human → Human approves → Tool completes → Agent responds

This is called a Long-Running Operation (LRO) - the tool needs to pause, wait for external input (human approval), then resume.

When to use Long-Running Operations:

💰 Financial transactions requiring approval (transfers, purchases)

🗑️ Bulk operations (delete 1000 records - confirm first!)

📋 Compliance checkpoints (regulatory approval needed)

💸 High-cost actions (spin up 50 servers - are you sure?)

⚠️ Irreversible operations (permanently delete account)

## What We're Building Today
Let's build a prescription coordinator agent with one tool that:

Pauses and asks for approval on prescription orders 
Completes or cancels based on the approval decision
This demonstrates the core long-running operation pattern: pause → wait for human input → resume.

## 3.1 The Prescription Tool with Approval Logic
Here's the complete function.

The ToolContext Parameter
Notice the function signature includes tool_context: ToolContext. ADK automatically provides this object when your tool runs. It gives you two key capabilities:

Request approval: Call tool_context.request_confirmation()
Check approval status: Read tool_context.tool_confirmation

In [16]:
def place_prescription_order(
   number : int, quality : str, tool_context: ToolContext
) -> dict:
    """Places a prescription order. Requires approval by a Doctor.

    Args:
        quality : Whether approved by Doctor or not

    Returns:
        Dictionary with prescription status
    """

    # -----------------------------------------------------------------------------------------------
    # SCENARIO 1 : This is the time this tool is called. Large orders need human approval - PAUSE here.
    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"⚠️ Prescrition : {number} . Do you want to approve?",
        )
        return {  # This is sent to the Agent
            "status": "pending",
            "message": f"This prescription requires approval",
        }

    # -----------------------------------------------------------------------------------------------
    # SCENARIO 2: The tool is called AGAIN and is now resuming. Handle approval response - RESUME here.
    if tool_context.tool_confirmation.confirmed:
        return {
            "status": "approved",
            "order_id": f"ORD-{number}-HUMAN",
            "quality": quality ,
            "message": f"Prescription approved",
        }
    else:
        return {
            "status": "rejected",
            "message": f"Prescription rejected",
        }


print("✅ Long-running functions created!")

✅ Long-running functions created!


How the Three Scenarios Work
The tool handles two scenarios by checking tool_context.tool_confirmation:

### Scenario 1: Prescription order - FIRST CALL

Tool detects it's a first call: if not tool_context.tool_confirmation:
Calls request_confirmation() to request human approval
Returns {'status': 'pending', ...} immediately
ADK automatically creates adk_request_confirmation event
Agent execution pauses - waiting for human decision

### Scenario 2: Prescription check - RESUMED CALL

Tool detects it's resuming: if not tool_context.tool_confirmation: is now False
Checks human decision: tool_context.tool_confirmation.confirmed
If True → Returns approved status
If False → Returns rejected status

Key insight: Between the two calls, your workflow code must detect the adk_request_confirmation event and resume with the approval decision

In [17]:
# Create shipping agent with pausable tool
confirming_agent = LlmAgent(
    name="confirming_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a Compounder , Doctors' coordinator assistant.
  
  When Doctor request to prescription:
   1. Use the place_prescription_order tool with the status of prescription 
   2. If the order status is 'pending', inform the user that approval is required
   3. After receiving the final result, provide a clear summary including:
      - Order status (approved/rejected)
      - Order ID (if available)
   4. Keep responses concise but informative
  """,
    tools=[FunctionTool(func=place_prescription_order)],
)

print("✅ Confirming Agent created!")

✅ Confirming Agent created!


## 3.2 Wrap in resumable App

The problem: A regular LlmAgent is stateless - each call is independent with no memory of previous interactions. If a tool requests approval, the agent can't remember what it was doing.

The solution: Wrap your agent in an App with resumability enabled. The App adds a persistence layer that saves and restores state.

What gets saved when a tool pauses:

All conversation messages so far
Which tool was called (place_prescription_order)
Tool parameters (5 , NotApproved)
Where exactly it paused (waiting for approval)
When you resume, the App loads this saved state so the agent continues exactly where it left off - as if no time passed.

In [18]:
# Wrap the agent in a resumable app - THIS IS THE KEY FOR LONG-RUNNING OPERATIONS!
prescription_app = App(
    name="prescription_coordinator",
    root_agent=confirming_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

print("✅ Resumable app created!")

✅ Resumable app created!


/tmp/ipykernel_13/1223343667.py:5: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=True),


## 3.3 Create Session and Runner with the App

Pass app=prescription_app instead of agent=... so the runner knows about resumability.

In [19]:
session_service = InMemorySessionService()

# Create runner with the resumable app
prescription_runner = Runner(
    app=prescription_app,  # Pass the app instead of the agent
    session_service=session_service,
)

print("✅ Runner created!")

✅ Runner created!


## 4 Building the Workflow¶
‼️ Important: The workflow code uses ADK concepts like Sessions, Runners, and Events. We'll cover what you need to know for long-running operations in this notebook.

⚠️ The Critical Part - Handling Events in Your Workflow
The agent won't automatically handle pause/resume. Every long-running operation workflow requires you to:

Detect the pause: Check if events contain adk_request_confirmation
Get human decision: In production, show UI and wait for user click. Here, we simulate it.
Resume the agent: Send the decision back with the saved invocation_id

Understand Key Technical Concepts
👉 events - ADK creates events as the agent executes. Tool calls, model responses, function results - all become events

👉 adk_request_confirmation event - This event is special - it signals "pause here!"

Automatically created by ADK when your tool calls request_confirmation()
Contains the invocation_id
Your workflow must detect this event to know the agent paused
👉 invocation_id - Every call to run_async() gets a unique invocation_id (like "abc123")

When a tool pauses, you save this ID
When resuming, pass the same ID so ADK knows which execution to continue
Without it, ADK would start a NEW execution instead of resuming the paused one

Helper Functions to Process Events

These handle the event iteration logic for you.

check_for_approval() - Detects if the agent paused

Loops through all events and looks for the special adk_request_confirmation event
Returns approval_id (identifies this specific request) and invocation_id (identifies which execution to resume)
Returns None if no pause detected

## 4.2 Helper Functions to Process Events
These handle the event iteration logic for us.

check_for_approval() - Detects if the agent paused

Loops through all events and looks for the special adk_request_confirmation event
Returns approval_id (identifies this specific request) and invocation_id (identifies which execution to resume)
Returns None if no pause detected

In [20]:
def check_for_approval(events):
    """Check if events contain an approval request.

    Returns:
        dict with approval details or None
    """
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if (
                    part.function_call
                    and part.function_call.name == "adk_request_confirmation"
                ):
                    return {
                        "approval_id": part.function_call.id,
                        "invocation_id": event.invocation_id,
                    }
    return None

print_agent_response() - Displays agent text

* Simple helper to extract and print text from events

In [21]:
def print_agent_response(events):
    """Print agent's text responses from events."""
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent > {part.text}")

create_approval_response() - Formats the human decision


* Takes the approval info and boolean decision (True/False) from the human
* Creates a FunctionResponse that ADK understands
* Wraps it in a Content object to send back to the agent


In [22]:
def create_approval_response(approval_info, approved):
    """Create approval response message."""
    confirmation_response = types.FunctionResponse(
        id=approval_info["approval_id"],
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(
        role="user", parts=[types.Part(function_response=confirmation_response)]
    )


print("✅ Helper functions defined")

✅ Helper functions defined


## 4.3 The Workflow Function - Let's tie it all together!
The run_prescription_workflow() function orchestrates the entire approval flow.

Look for the code explanation in the cell below.

In [23]:
async def run_prescription_workflow(query: str, auto_approve: bool = True):
    """Runs  prescription workflow  with approval handling.

    Args:
        query: User's prescription request
        auto_approve: Whether to auto-approve large orders (simulates human decision)
    """

    print(f"\n{'='*60}")
    print(f"User > {query}\n")

    # Generate unique session ID
    session_id = f"order_{uuid.uuid4().hex[:8]}"

    # Create session
    await session_service.create_session(
        app_name="prescription_coordinator", user_id="test_user", session_id=session_id
    )

    query_content = types.Content(role="user", parts=[types.Part(text=query)])
    events = []

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 1: Send initial request to the Agent. If num_containers > 5, the Agent returns the special `adk_request_confirmation` event
    async for event in prescription_runner.run_async(
        user_id="test_user", session_id=session_id, new_message=query_content
    ):
        events.append(event)

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 2: Loop through all the events generated and check if `adk_request_confirmation` is present.
    approval_info = check_for_approval(events)

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 3: If the event is present, it's a large order - HANDLE APPROVAL WORKFLOW
    if approval_info:
        print(f"⏸️  Pausing for approval...")
        print(f"🤔 Human Decision: {'APPROVE ✅' if auto_approve else 'REJECT ❌'}\n")

        # PATH A: Resume the agent by calling run_async() again with the approval decision
        async for event in prescription_runner.run_async(
            user_id="test_user",
            session_id=session_id,
            new_message=create_approval_response(
                approval_info, auto_approve
            ),  # Send human decision here
            invocation_id=approval_info[
                "invocation_id"
            ],  # Critical: same invocation_id tells ADK to RESUME
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent > {part.text}")

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    else:
        # PATH B: If the `adk_request_confirmation` is not present - no approval needed - order completed immediately.
        print_agent_response(events)

    print(f"{'='*60}\n")


print("✅ Workflow function ready")

✅ Workflow function ready


Code breakdown

**Step 1: Send initial request to the Agent**

* Call run_async() to start agent execution
* Collect all events in a list for inspection
* 
**Step 2: Detect Pause**

* Call check_for_approval(events) to look for the special event: adk_request_confirmation
* Returns approval info (with invocation_id) if the special event is present; None if completed

**Step 3: Resume execution**

PATH A:

* If the approval info is present, at this point the Agent pauses for human input.
* Once the Human input is available, call the agent again using run_async() and pass in the Human input.
* Critical: Same invocation_id (tells ADK to RESUME, not restart)
* Display agent's final response after resuming

**PATH B:**

* If the approval info is not present, then approval is not needed and the agent completes execution.

In [24]:
from google.adk.runners import InMemoryRunner

# Initialize the runner and keep it in memory
#prescription_runner = InMemoryRunner(agent=root_agent)
#print("✅ Runner is initialized and ready.")

## 4.4 Demo: Testing the Workflow
Now, let's run our demos. Notice how much cleaner and easier to read they are. All the complex logic for pausing and resuming is now hidden away in our run_workflow helper function, allowing us to focus on the tasks we want the agent to perform.

Note: You may see warnings like Warning: there are non-text parts in the response: ['function_call'] - this is normal and can be ignored. It just means the agent is calling tools in addition to generating text.

In [25]:
# Demo : Workflow simulates human decision: APPROVE ✅
await run_prescription_workflow("Prepare prescription 1 , approved", auto_approve=True)


User > Prepare prescription 1 , approved



/usr/local/lib/python3.11/dist-packages/google/adk/tools/tool_context.py:92: UserWarning: [EXPERIMENTAL] ToolConfirmation: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  ToolConfirmation(
/usr/local/lib/python3.11/dist-packages/google/adk/agents/invocation_context.py:298: UserWarning: [EXPERIMENTAL] BaseAgentState: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self.agent_states[event.author] = BaseAgentState()


⏸️  Pausing for approval...
🤔 Human Decision: APPROVE ✅

Agent > I have prepared your prescription with order ID ORD-1-HUMAN and it has been approved.



##  4.5 Run your agent
Now it's time to bring your agent to life and send it a query. To do this, you need a Runner, which is the central component within ADK that acts as the orchestrator. It manages the conversation, sends our messages to the agent, and handles its responses.

**a. Create an InMemoryRunner and tell it to use our root_agent:**

In [26]:
runner = InMemoryRunner(agent=froot_agent)

print("✅ Runner created.")

✅ Runner created.


## 4.6 Add LoggingPlugin to Runner
The following code creates the InMemoryRunner. This is used to programmatically invoke the agent.

To use LoggingPlugin in the above research agent, 1) Import the plugin 2) Add it when initializing the InMemoryRunner.

### ADK's built-in LoggingPlugin
But you don't have to define all the callbacks and plugins to capture standard Observability data in ADK. Instead, ADK provides a built-in LoggingPlugin that automatically captures all agent activity:

🚀 User messages and agent responses

⏱️ Timing data for performance analysis

🧠 LLM requests and responses for debugging

🔧 Tool calls and results

✅ Complete execution traces

In [27]:
from google.adk.runners import InMemoryRunner
from google.adk.plugins.logging_plugin import (
    LoggingPlugin,
)  # <---- 1. Import the Plugin
from google.genai import types
import asyncio

runner = InMemoryRunner(
    agent=froot_agent,
    plugins=[
        LoggingPlugin()
    ],  # <---- 2. Add the plugin. Handles standard Observability logging across ALL agents
)

print("✅ Runner configured")

✅ Runner configured


b. Now you can call the .run_debug() method to send our prompt and get an answer.

👉 This method abstracts the process of session creation and maintenance and is used in prototyping.

In [28]:
response = await runner.run_debug("Medicines for Throat inflamation")


 ### Created new session: debug_session_id

User > Medicines for Throat inflamation
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-8b0e7f1b-658a-470d-a017-bc2a52784fc5
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: PrescriptionPipeline
[logging_plugin]    User Content: text: 'Medicines for Throat inflamation'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-8b0e7f1b-658a-470d-a017-bc2a52784fc5
[logging_plugin]    Starting Agent: PrescriptionPipeline
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: PrescriptionPipeline
[logging_plugin]    Invocation ID: e-8b0e7f1b-658a-470d-a017-bc2a52784fc5
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: ResearchSystem
[logging_plugin]    Invocation ID: e-8b0e7f1b-658a-470d-a017-bc2a52784fc5
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin

[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: RefinerAgent
[logging_plugin]    Content: function_call: exit_loop
[logging_plugin]    Token Usage - Input: 5053, Output: 10
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: f3d5f0fe-fd32-478c-b6a8-2aa8c917f4fa
[logging_plugin]    Author: RefinerAgent
[logging_plugin]    Content: function_call: exit_loop
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['exit_loop']
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: exit_loop
[logging_plugin]    Agent: RefinerAgent
[logging_plugin]    Function Call ID: adk-c8ed89e3-364c-4644-a826-3dbfbdeaa473
[logging_plugin]    Arguments: {}
[logging_plugin] 🔧 TOOL COMPLETED
[logging_plugin]    Tool Name: exit_loop
[logging_plugin]    Agent: RefinerAgent
[logging_plugin]    Function Call ID: adk-c8ed89e3-364c-4644-a826-3dbfbdeaa473
[logging_plugin]    Result: {'status': 'approved', 'message': 'Prescription approved. Exiting refinement loo

## 5 Get Prescription Printed

In [29]:
from IPython.display import HTML, Markdown, display

# The response variable is a list that contains a mix of strings and Event objects.
# We must convert every item to a string before joining.
string_response_list = [str(item) for item in response]

# Now, join the list of strings.
full_response_text = "\n".join(string_response_list)

# Display the full string as Markdown.
display(Markdown(full_response_text))

model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""Here are some Ayurvedic medicines and their dosages for throat inflammation:

*   **Turmeric Milk (Haldi Doodh):** Mix 1 teaspoon of turmeric powder in a glass of warm milk. Drink once daily, preferably before bedtime.
*   **Licorice Root (Mulethi/Yashtimadhu):**
    *   Take 1 teaspoon of licorice powder mixed with warm water or honey, twice daily.
    *   Alternatively, chew a small stick of licorice root or suck on licorice lozenges.
    *   For gargling, boil 1 teaspoon of licorice powder in a cup of warm water and use the solution.
*   **Tulsi (Holy Basil) Tea:** Boil 10-15 fresh tulsi leaves in 2 cups of water until reduced to 1 cup. Drink this tea twice daily.
*   **Triphala Powder:** Take 1 teaspoon of Triphala powder in warm water. Gargle with the solution twice a day, or take it orally once a day.
*   **Honey and Ginger:** Mix 1 teaspoon of fresh ginger juice with 1 teaspoon of honey. Take this mixture 2-3 times daily.
*   **Talisadi Churna:** Take 1-2 grams with honey or warm water after meals.
*   **Sitopaladi Churna:** Take 1-2 grams with honey, 2-3 times a day.
*   **Khadiradi Vati:** Suck 1-2 tablets slowly, 3-4 times a day.
*   **Turmeric and Salt Gargle:** Mix ½ teaspoon of turmeric powder and a pinch of salt in warm water. Gargle 2-3 times a day."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        title='all-cures.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEfrVe2CiskAybbCmniTeWQzjj92vql2TCTIEJofpGrFQa4rK3po6suqyYGZCkVGi4zDzi2nXBCaLpOVZSyLq3JuZEBlnWcjPxn4mUr-FwwDkDZpkiVzeeSzmJ2j71Fr-zEBBk9zZzDAYrdcT0MZHD_6qz9JhVi7QgxrSPUMfJd8IBtQCu39qdo'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='netmeds.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGb9LYUGQYILTEAdDAFfj1ZGmD3j_7zr-eB4lW4eY2Vh-FCeoKxo684a_xQNQZldT_Pf-gkWzKu4ONOMmKFBJGZc_wLvkMKF068t54vonwVe24REiGVUfyju3WlYAmJvQ3gcU5XxJtlBPKmLMOWYG6aRXhhge6LfQwz4qw7Ud3gnrqmsJWKksWZ6_0U3opSL-IgsKiF-9Y6Lmc9BXME0p3TtOMCSuUbuMC6J2cXagU3JXLkAbupAXj3HgG6yiLm6uR3HYIsRp3OgkKURstZNQ=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='ask-ayurveda.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHAdxYtmNvd2HL4iJ_8u3sEi9qH-mc8FSVU433Hf9GqgoxU34L_4XCDHNje-GjvxLoD222QWVjBQWN_PfE1sW4A2wbMpJtafcNb5lyvoTzrkrwIIN5QMO_FoKOTjAk8GS5ypJzH9K47R0zEBAlylnw5vRqbKdQN6C0Q78uYVKu4v4eDzkAgdGH6KbsmxkgP-wMqmn11B2zVb54PgXk='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='aatreyaayurved.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHrj8styUNoVTpS8DF4f5MyZBQ9MIVoWPT8lpkPUTfMIpkLUst9n0atmBtRkfJjbZkqZqI6GqSHkJ1wD1nIhAoO100ZFIIkLK1Q56rxdMBUTZeqse85Vx3SpTOxrTsnXerx95TNLJqYE5fubQwryWWeS3xHylh-R7VBkieXQWEdA_s24CR78xySBe7NXcfw'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='travancoreayurveda.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZqIEsJgCEbdZAhxmqUyBLsUA_PCuIolXPpxotLEsdORJtYI5PvyQYWklon78HCUW2PE9qcvlfLXJUT9F2trXZY6p5TKZYuoLzscD_H5R7WU6PF9xsFgJY2_Ry5FJrJHcTLyQ4DddKUypgenxOgdpJ-27-noXpn6lnIt2xug9AhoMapN-hqmjn'
      )
    ),
    <... 7 more items ...>,
  ],
  grounding_supports=[
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
        2,
      ],
      segment=Segment(
        end_index=218,
        start_index=174,
        text='Drink once daily, preferably before bedtime.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        3,
        4,
      ],
      segment=Segment(
        end_index=351,
        start_index=219,
        text="""*   **Licorice Root (Mulethi/Yashtimadhu):**
    *   Take 1 teaspoon of licorice powder mixed with warm water or honey, twice daily."""
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        1,
        2,
        5,
      ],
      segment=Segment(
        end_index=440,
        start_index=356,
        text='*   Alternatively, chew a small stick of licorice root or suck on licorice lozenges.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
      ],
      segment=Segment(
        end_index=542,
        start_index=445,
        text='*   For gargling, boil 1 teaspoon of licorice powder in a cup of warm water and use the solution.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
      ],
      segment=Segment(
        end_index=675,
        start_index=648,
        text='Drink this tea twice daily.'
      )
    ),
    <... 6 more items ...>,
  ],
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHU1kvyf3g4HWzqcrhFOVKaWAIkJB2m3d2EeaFdMruLEfaPJMEDSIwOmrvYtWwg5YVMF9rbPqphMZJUdTqLE6Els_d8EzxZ1_Q_ozPR0x4cjYa5jI6-bDFy2mVMrk1gQTp5vrvqGe_jXRKHT3G7UMOJz5FcBFCFMyyRRIaAQ5rjJJmcyojAXlu27JKfobAMkbxdAU8tB6I6pyTi93AlK-8qwITDtcbx7wo4dnPLmXmhxS3of8pqqk8EbmZO4kBn">Effective Ayurvedic treatments for pharyngitis with dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHBxYJzvxvRijKYkWxxiWHfr0BV1IMyDJPOugUCIOYrHcPLcWn2u_umlMHUm7E890PcQfq5cWUNWW6WuSYE4z8gkfWw3ZzCaqIhrfwt05tsiSROG7QbCvb30Jw_YGpKgdtXjfFvCsfZW4W6avzEqn_6RjESacmPCZViR7Kgq02RMf6lfnAU3di5ldTDRpZc9IW2dPPJSZaVFrgGSBLVjf2iSwxOjrn6Ig2NXuWgYgvtrFZ9aHwJEQ==">Ayurvedic medicines for throat inflammation dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFIchyxOe8C5JPyCNNGiucHJfVefd6eKKIECsqiLwAB2MWfp8Y1BkQOJ3MT62X1KDaN6_-woITWarU2c9Uvif1G29xwxXms0ABuCvzl7NK9O10jEF9y3pi3t-RNt1-bc4zbHqoSBYEbXEWVR7J6wAtx-AiYgGqbwAPwXcGPy76XhilH-JyXCoJStuOjPBZqNZjk6Q-pdY4foTI5qSTFRQ3jv0KvlBtTImOopd-4m2Bop9g=">Ayurvedic remedies for sore throat with dosage</a>
  </div>
</div>
"""
  ),
  web_search_queries=[
    'Ayurvedic medicines for throat inflammation dosage',
    'Ayurvedic remedies for sore throat with dosage',
    'Effective Ayurvedic treatments for pharyngitis with dosage',
  ]
) partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=407,
  prompt_token_count=67,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=67
    ),
  ],
  tool_use_prompt_token_count=108,
  tool_use_prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=108
    ),
  ],
  total_token_count=582
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='AyurvedicAgent' actions=EventActions(skip_summarization=None, state_delta={'ayurvedic_findings': 'Here are some Ayurvedic medicines and their dosages for throat inflammation:\n\n*   **Turmeric Milk (Haldi Doodh):** Mix 1 teaspoon of turmeric powder in a glass of warm milk. Drink once daily, preferably before bedtime.\n*   **Licorice Root (Mulethi/Yashtimadhu):**\n    *   Take 1 teaspoon of licorice powder mixed with warm water or honey, twice daily.\n    *   Alternatively, chew a small stick of licorice root or suck on licorice lozenges.\n    *   For gargling, boil 1 teaspoon of licorice powder in a cup of warm water and use the solution.\n*   **Tulsi (Holy Basil) Tea:** Boil 10-15 fresh tulsi leaves in 2 cups of water until reduced to 1 cup. Drink this tea twice daily.\n*   **Triphala Powder:** Take 1 teaspoon of Triphala powder in warm water. Gargle with the solution twice a day, or take it orally once a day.\n*   **Honey and Ginger:** Mix 1 teaspoon of fresh ginger juice with 1 teaspoon of honey. Take this mixture 2-3 times daily.\n*   **Talisadi Churna:** Take 1-2 grams with honey or warm water after meals.\n*   **Sitopaladi Churna:** Take 1-2 grams with honey, 2-3 times a day.\n*   **Khadiradi Vati:** Suck 1-2 tablets slowly, 3-4 times a day.\n*   **Turmeric and Salt Gargle:** Mix ½ teaspoon of turmeric powder and a pinch of salt in warm water. Gargle 2-3 times a day.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch='ParallelResearchTeam.AyurvedicAgent' id='f40ad53f-fc73-43d3-abd5-7c1b33bebd78' timestamp=1763451095.249493
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""For throat inflammation, the following homeopathic medicines are commonly recommended, along with their typical dosages:

*   **Belladonna:** This is often indicated for acute throat infections with a sudden onset, high fever, a bright red and swollen throat, and a raw, burning sensation. Symptoms may be worse on the right side, with a flushed face and thirst for cold drinks.
    *   **Dosage:** In acute conditions with high fever, Belladonna can be taken every hour or two for 2-3 doses. If there's no improvement after a few doses, another remedy may be considered. For general throat issues, ThroatCalm tablets contain Belladonna 3C HPUS and can be taken as directed on the package.

*   **Hepar sulph:** This remedy is useful when there's a sensation of a splinter or fishbone stuck in the throat. Symptoms are often worse after exposure to cold and may be temporarily relieved by hot drinks.
    *   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours. ThroatCalm tablets containing Hepar Sulphur can be taken as directed.

*   **Mercurius solubilis (Merc sol):** This is frequently indicated for acute sore throats and tonsillitis with a burning and raw sensation, tightness, and a feeling of a foreign body in the throat. The throat may appear bluish-red and swollen, with excessive salivation and an offensive taste or odor in the mouth.
    *   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours. ThroatCalm tablets containing Mercurius solubilis 4C HPUS can be taken as directed.

*   **Phytolacca decandra:** This remedy is recommended for intense sore throats accompanied by a feeling of a lump or splinter, with pain that may radiate to the ears. The tonsils might appear dark red or bluish-red and swollen, with excessive pain and a burning sensation when swallowing.
    *   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours. ThroatCalm tablets containing Phytolacca decandra 3C HPUS can be taken as directed.

It's important to note that homeopathic treatment is highly individualized. The selection of a remedy depends on the totality of symptoms presented by the patient. If symptoms are severe or do not improve, consulting a qualified homeopathic practitioner is recommended. For general sore throat relief, some products like ThroatCalm suggest dissolving a specific number of tablets every 15 minutes for the first hour, then every 6 hours, decreasing frequency with improvement."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        title='skolahomeopatie.cz',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHRu65-Cu1Bvv3sOQnYzRlQ3lW8oH3G4zRShKYi8WLsoQ2lIwNvdm6s4mRPZxWEj8UYm2qM46VdttPpd1B5q1UncDI9nx5RpYiF2t8Ra_trR-GHzvNiXWiBwQx8lCIBQrMBuYFBJQUsDs6gTwHKQ4zErh6TqGvplU3yP4qMz3lg8sxrF2dX8jpqsnJsEgajlb4qjJUbGIOBH_yc2iHPAjfposfw39gXydfm9xS69_VfcEnkFFKQww4='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='karenleadbeater.co.uk',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH7k01ihfjbrqMOKGF0C5FOiNu6AyNPGudnjV9fs7vEFkob5gv8E7_YfLGjee0clxLqX2dPBNd3mkSahx6NUfe-hQokOSFHhgx4Z5ZhV0jzWgDJzR0cpEeC5gntk4Kv3u53vLkJuj0vpQkWEtCujhR61dW1Ik6MQpUYwVPYL9otDULfQqx_gbv1t_M7we7z1PFZzLGB'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='oscillo.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQP9iVBwW3bf9bYjRxM3l3zhJc5C8ptG7y8852RhULfiiNddsxi6X9kEo0-1GT_kstv3zOZ7zl2mP5TKktaffcnlS0KnwFyGCEp95QwSZv0zH4wkyRRB2tRmrfz_jd8LuuCZpI-vg7_sAcsKVN'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='delhi.gov.in',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6-iQWf0C9KrOJseJpK5eKGXoeJqEKRTdw5c4z23cVCtBdbwf9Dg1pkzi3MKrWL0Smm0zEUw-OWPrClHv77AMtvPyP5ZDR4aOZL0hgbGped6i2lmFmtpHZhJyyXOG06ndTzqAf34yqeHvwpQnK1yXp3cDtBIXIiBY99xFwiixclPdOuvNvYpMnRA3NwaoRk0IIJa7uRb09ONzLGDg9'
      )
    ),
  ],
  grounding_supports=[
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
      ],
      segment=Segment(
        end_index=571,
        start_index=493,
        text="If there's no improvement after a few doses, another remedy may be considered."
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        2,
      ],
      segment=Segment(
        end_index=689,
        start_index=572,
        text='For general throat issues, ThroatCalm tablets contain Belladonna 3C HPUS and can be taken as directed on the package.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        3,
      ],
      segment=Segment(
        end_index=987,
        start_index=905,
        text='*   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        2,
      ],
      segment=Segment(
        end_index=1057,
        start_index=988,
        text='ThroatCalm tablets containing Hepar Sulphur can be taken as directed.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        3,
      ],
      segment=Segment(
        end_index=1462,
        start_index=1380,
        text='*   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours.'
      )
    ),
    <... 5 more items ...>,
  ],
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHIoDNuN_GJPQ5V1yo-wZ9AnO0gdojADvmU48yCYMfTv37LknVvWLc0FUT49xTsd3UIihS8Gx2oOhyeVY9A7tqVIU3CR6jZbxtV3wzSpTnnb6ofvfzq2BaAdK641YKuz0GdrT8ExsOSg2i5EQ3X0ZvYG4Pj8uCs9Rq8q4eRfL0FfEGOcBPOyeCjid7gkMH9hWxuGxlFGRjFGzVDiQmvXfcXn3QFl6BqL-GuztsOEucSQ8cY">homeopathic remedies for sore throat with fever</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFfq14PWFJDGWD9HAUzZsxPOilRQGTUM9oqUqimOQ-OGoyyChV6gglJwwWbXP77PDcn__SrfmVixrKSwVNI0myIajALnbDWSAc7r3HGGkEjZi0QBDLmrVrQFMuOVKZzw4LJcpsCiCuDBjNohFMTJlfPnt1V5gj7luZstwveBQeLL_BWJwHkW0MW6QhZciO7I4hOA4KUo5RnSWxNdwZXH1mVpRvFDcLMQsDE6oCN-88RJnNDdafldKw=">homeopathic medicine for throat inflammation dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHL2hiaGwfIjVxewJjrCtRZ2Qzzr-69psuhnFZrhEAZ37ccnn7h42qLuoS6NGn7SQeHZ5dXeocYTdxSdZq-VdCweQX9qLwSK1iNLZDAkKumxDb9L8ZD8ppmmAAIAv8U1nHvalIg5b2f6hgtQE3tT6qzrnTfImlJzmrE-zFaSIJ6C-BSjeYxL4B70loTbJnxdNbeh2DwyXqeaKTYxqEmO0KSGi7vwyxM7giqNZq45OSf">homeopathic treatment for tonsillitis dosage</a>
  </div>
</div>
"""
  ),
  web_search_queries=[
    'homeopathic medicine for throat inflammation dosage',
    'homeopathic remedies for sore throat with fever',
    'homeopathic treatment for tonsillitis dosage',
  ]
) partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=598,
  prompt_token_count=68,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=68
    ),
  ],
  tool_use_prompt_token_count=106,
  tool_use_prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=106
    ),
  ],
  total_token_count=772
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='HomeopathicAgent' actions=EventActions(skip_summarization=None, state_delta={'homeopathic_findings': "For throat inflammation, the following homeopathic medicines are commonly recommended, along with their typical dosages:\n\n*   **Belladonna:** This is often indicated for acute throat infections with a sudden onset, high fever, a bright red and swollen throat, and a raw, burning sensation. Symptoms may be worse on the right side, with a flushed face and thirst for cold drinks.\n    *   **Dosage:** In acute conditions with high fever, Belladonna can be taken every hour or two for 2-3 doses. If there's no improvement after a few doses, another remedy may be considered. For general throat issues, ThroatCalm tablets contain Belladonna 3C HPUS and can be taken as directed on the package.\n\n*   **Hepar sulph:** This remedy is useful when there's a sensation of a splinter or fishbone stuck in the throat. Symptoms are often worse after exposure to cold and may be temporarily relieved by hot drinks.\n    *   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours. ThroatCalm tablets containing Hepar Sulphur can be taken as directed.\n\n*   **Mercurius solubilis (Merc sol):** This is frequently indicated for acute sore throats and tonsillitis with a burning and raw sensation, tightness, and a feeling of a foreign body in the throat. The throat may appear bluish-red and swollen, with excessive salivation and an offensive taste or odor in the mouth.\n    *   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours. ThroatCalm tablets containing Mercurius solubilis 4C HPUS can be taken as directed.\n\n*   **Phytolacca decandra:** This remedy is recommended for intense sore throats accompanied by a feeling of a lump or splinter, with pain that may radiate to the ears. The tonsils might appear dark red or bluish-red and swollen, with excessive pain and a burning sensation when swallowing.\n    *   **Dosage:** For tonsillitis, 3 globules of size 30 may be taken every 3 hours. ThroatCalm tablets containing Phytolacca decandra 3C HPUS can be taken as directed.\n\nIt's important to note that homeopathic treatment is highly individualized. The selection of a remedy depends on the totality of symptoms presented by the patient. If symptoms are severe or do not improve, consulting a qualified homeopathic practitioner is recommended. For general sore throat relief, some products like ThroatCalm suggest dissolving a specific number of tablets every 15 minutes for the first hour, then every 6 hours, decreasing frequency with improvement."}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch='ParallelResearchTeam.HomeopathicAgent' id='b5985326-3363-4367-82f4-71dc428e67d9' timestamp=1763451095.435546
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""Here are some allopathic medicines commonly used for throat inflammation, along with their typical dosages. Please note that a healthcare professional should always be consulted for a proper diagnosis and personalized treatment plan.

**Over-the-Counter (OTC) Pain Relievers and Anti-inflammatories:**

*   **Ibuprofen (e.g., Advil, Motrin):** This non-steroidal anti-inflammatory drug (NSAID) helps reduce pain and inflammation.
    *   **Dosage for Adults:** Typically, 200 mg to 400 mg every 4 to 6 hours as needed, not to exceed 1200 mg in 24 hours for OTC use. Higher doses may be prescribed by a doctor.
*   **Acetaminophen (e.g., Tylenol):** This medication is effective for pain relief and fever reduction but does not reduce inflammation.
    *   **Dosage for Adults:** Generally, 325 mg to 1000 mg every 4 to 6 hours as needed, not to exceed 4000 mg in 24 hours.
*   **Naproxen (e.g., Aleve):** Another NSAID that reduces pain and inflammation.
    *   **Dosage for Adults:** Typically, 220 mg to 550 mg every 12 hours.

**Prescription Medications (for bacterial infections like Strep Throat):**

*   **Amoxicillin:** A penicillin-type antibiotic often prescribed for bacterial throat infections.
    *   **Dosage for Adults (mild to moderate infections):** 500 mg every 12 hours or 250 mg every 8 hours for 10 days.
    *   **Dosage for Adults (severe infections):** 875 mg every 12 hours or 500 mg every 8 hours for 10 days.
    *   **Dosage for Children:** 50 mg/kg/day in 2 or 3 divided doses for 10 days. or 50 mg/kg once daily (max 1000 mg) for 10 days.
*   **Penicillin V:** Another common antibiotic for bacterial throat infections.
    *   **Dosage for Adults:** 500 mg twice daily or 250 mg four times daily for 10 days.
    *   **Dosage for Children:** 250 mg two or three times daily for 10 days (dose adjusted by weight).
*   **Dexamethasone:** A corticosteroid that can help reduce severe inflammation and pain.
    *   **Dosage for Adults:** A single oral dose of 10 mg is often recommended.
    *   **Dosage for Children (5-18 years):** 0.6 mg/kg with a maximum of 10 mg as a single dose.

**Important Considerations:**

*   **Antibiotics** are only effective against bacterial infections and are not useful for viral infections, which are the most common cause of sore throats.
*   **Throat sprays and lozenges** containing local anesthetics, antiseptics, or anti-inflammatories can provide temporary relief.
*   Always consult a healthcare provider to determine the cause of your throat inflammation and the most appropriate treatment. Self-medication can be risky."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        title='drugs.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGDLCTU-QLpDaaOUj1N4DGqqonz4OqzmNYn3OyZPHMXC7rjuZnIwfFJNzP8UUNvHEwA6jQD93Y7qRUI248TFOJTmW_FujIvxD_rPe8a0aoNWSdQ3fpwIIqIa8WfVFIZ68hy8w2dzqvE43kzjD3QpEIkCDlDh3u2oNvrCtwBicrFcGcMTdTUOkQz'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='verywellhealth.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGq2P3iFL1j9LbmYj8xz9kvs7MZa0C0ooKTSpUg7Gv8recAo7Cz6YUNsL8XySDj8q9nqiPhLsRCJFx4yjvXvWTyOVSZqKzemHt0MkIHYO7ohaPnZ7UUB5ZjaXwa5_viNlFGd4Hkag5kjV0T8u2_fU-7AXepjQz0g0QNiWk-TjBhrEwMjw=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='droracle.ai',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHOVikihYbH0QESPgw8YYPGWvyk99nhvFqJzxidyEzMenStrCJK6QmLrj5Gf8fLnFKa9YSoGwHc-ecH2KiMCpQxxwCv__nHm6LqnZlt2VhaMgYu2YLeW9E_46PBzjrlhV4oNzYu9qM-rL0Pb-g8ZE2hDBMLkbvfsfe10r6QOgXGbhZE4TBWXI-1S6Xdoq4h-KlgKMIjBuALEZzbBfENtCBQKFrdDdiD-Gn3lzzE0hBmjHJhEkXQJi3NBkS2xsvXSN2DJCYUtd2ExcQVjOezt7Ic45ysGw=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='medicalnewstoday.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGY8Jau807Q6nbOMQFA-bW8Pwi5mloQEe0-YkZevk5xgYSqzAz0HYTCdXT_Q-Jhxzs_IC9G9IrXGO1yiNW9wXPy_c4-BtJD75hHUfhA14xq3N5vMU6kfarS4uRZY2aZgsa1PeyXYaeL1fVW9AQaem5_LzUb7v8QXv1MCfEr'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='medscape.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHx0cXu6JhXPpwacXOntrwnUl-H2cDmk5jQNXh4C64IvV2Cf2kUElqWLFeaR97FU2vskWnuOGvcpwJsC5fLOTAuCkTXqSBANd3sDjbDPvsWZ9t-iHLMianO5r1dnkZQ5-YnDkKvXIAYUP9EIwe2YMQWwQ=='
      )
    ),
    <... 5 more items ...>,
  ],
  grounding_supports=[
    GroundingSupport(
      grounding_chunk_indices=[
        0,
      ],
      segment=Segment(
        end_index=609,
        start_index=566,
        text='Higher doses may be prescribed by a doctor.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
      ],
      segment=Segment(
        end_index=872,
        start_index=752,
        text='*   **Dosage for Adults:** Generally, 325 mg to 1000 mg every 4 to 6 hours as needed, not to exceed 4000 mg in 24 hours.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        1,
      ],
      segment=Segment(
        end_index=1029,
        start_index=959,
        text='*   **Dosage for Adults:** Typically, 220 mg to 550 mg every 12 hours.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        2,
        3,
      ],
      segment=Segment(
        end_index=1326,
        start_index=1211,
        text='*   **Dosage for Adults (mild to moderate infections):** 500 mg every 12 hours or 250 mg every 8 hours for 10 days.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        2,
      ],
      segment=Segment(
        end_index=1436,
        start_index=1331,
        text='*   **Dosage for Adults (severe infections):** 875 mg every 12 hours or 500 mg every 8 hours for 10 days.'
      )
    ),
    <... 8 more items ...>,
  ],
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGl3yJCtZWKewEWPrjMKFaDxpyHN3hLFySd94PTqki5hR-kEfDUhGviIbpC6rLhGDLhmVeS9AOLFiX6dgqQiw1ZKBHPn1q0sIXb6wRQjyl5h-19azzC8pZo709UYgW74DQJysQAjP5tb6ylM7ScvGpEkUqf5gVy1kqjJZd40kYjns9r6VLq06l8SCm99qhUuFkbSHgXpPUL88CiT8HuLQ==">sore throat treatment dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFcRV1-dTbetMYZo8BWGmXswVWP2hA3Wh4ACT_UjGw_paWK_H8WKdcFSBB6IRydvE771L_JiY5rKVHED_h_OTqeeMu66nA1CZ9uT3bGPie2DlKcHOodGYMTb-5xM9g6GhTrk7gjfnXzgnOXxfm6pWIWdMsK5TCYUpdRqbakj6Bzg0AGmQy_-Dfkz4uiI3fkcONPegnGKA4WnpOonIkVAGf3EtBsxqCAQcr-cVBmVO7J71ePUq9P">allopathic medicine for throat inflammation dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHiIloADr6Oi4dwNnPt_Z_hcXN02CjYxHGH3Ovu78DH7cH49RK2SfRzS1Tm_JRhmTN8o9kbqb4Gv5oCBENUOZyNYlz70CDnS5J_o30QOtoZGtdPGSQfGqGcSzKsPG9-bz9WdWeXcvTjeFVN3_80EDL4ZGmPeFweyajFkQd69HsD0DPA8Ez2-scsyIcS0viDI-_zafiJMNTBcnsIeJP54u57zvLEWCi4mYLhgjbECc5ryCHFPLzbkQ==">over the counter medication for throat inflammation</a>
  </div>
</div>
"""
  ),
  web_search_queries=[
    'allopathic medicine for throat inflammation dosage',
    'sore throat treatment dosage',
    'over the counter medication for throat inflammation',
  ]
) partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=754,
  prompt_token_count=68,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=68
    ),
  ],
  tool_use_prompt_token_count=102,
  tool_use_prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=102
    ),
  ],
  total_token_count=924
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='AllopathicAgent' actions=EventActions(skip_summarization=None, state_delta={'allopathic_findings': 'Here are some allopathic medicines commonly used for throat inflammation, along with their typical dosages. Please note that a healthcare professional should always be consulted for a proper diagnosis and personalized treatment plan.\n\n**Over-the-Counter (OTC) Pain Relievers and Anti-inflammatories:**\n\n*   **Ibuprofen (e.g., Advil, Motrin):** This non-steroidal anti-inflammatory drug (NSAID) helps reduce pain and inflammation.\n    *   **Dosage for Adults:** Typically, 200 mg to 400 mg every 4 to 6 hours as needed, not to exceed 1200 mg in 24 hours for OTC use. Higher doses may be prescribed by a doctor.\n*   **Acetaminophen (e.g., Tylenol):** This medication is effective for pain relief and fever reduction but does not reduce inflammation.\n    *   **Dosage for Adults:** Generally, 325 mg to 1000 mg every 4 to 6 hours as needed, not to exceed 4000 mg in 24 hours.\n*   **Naproxen (e.g., Aleve):** Another NSAID that reduces pain and inflammation.\n    *   **Dosage for Adults:** Typically, 220 mg to 550 mg every 12 hours.\n\n**Prescription Medications (for bacterial infections like Strep Throat):**\n\n*   **Amoxicillin:** A penicillin-type antibiotic often prescribed for bacterial throat infections.\n    *   **Dosage for Adults (mild to moderate infections):** 500 mg every 12 hours or 250 mg every 8 hours for 10 days.\n    *   **Dosage for Adults (severe infections):** 875 mg every 12 hours or 500 mg every 8 hours for 10 days.\n    *   **Dosage for Children:** 50 mg/kg/day in 2 or 3 divided doses for 10 days. or 50 mg/kg once daily (max 1000 mg) for 10 days.\n*   **Penicillin V:** Another common antibiotic for bacterial throat infections.\n    *   **Dosage for Adults:** 500 mg twice daily or 250 mg four times daily for 10 days.\n    *   **Dosage for Children:** 250 mg two or three times daily for 10 days (dose adjusted by weight).\n*   **Dexamethasone:** A corticosteroid that can help reduce severe inflammation and pain.\n    *   **Dosage for Adults:** A single oral dose of 10 mg is often recommended.\n    *   **Dosage for Children (5-18 years):** 0.6 mg/kg with a maximum of 10 mg as a single dose.\n\n**Important Considerations:**\n\n*   **Antibiotics** are only effective against bacterial infections and are not useful for viral infections, which are the most common cause of sore throats.\n*   **Throat sprays and lozenges** containing local anesthetics, antiseptics, or anti-inflammatories can provide temporary relief.\n*   Always consult a healthcare provider to determine the cause of your throat inflammation and the most appropriate treatment. Self-medication can be risky.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch='ParallelResearchTeam.AllopathicAgent' id='ffe605cc-98da-4120-a0c3-df0146147828' timestamp=1763451095.625222
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""**Executive Prescription for Throat Inflammation:**

For pain/fever, consider Ibuprofen 200-400mg q4-6h, Acetaminophen 325-1000mg q4-6h, or Naproxen 220-550mg q12h. For bacterial infections, Amoxicillin or Penicillin V. Homeopathic options include Belladonna, Hepar sulph, Merc sol, or Phytolacca. Ayurvedic remedies: Turmeric milk, Licorice, Tulsi tea, Honey/Ginger.

This is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes."""
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=142,
  prompt_token_count=3450,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3450
    ),
  ],
  total_token_count=3592
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='AggregatorAgent' actions=EventActions(skip_summarization=None, state_delta={'prescription_summary': '**Executive Prescription for Throat Inflammation:**\n\nFor pain/fever, consider Ibuprofen 200-400mg q4-6h, Acetaminophen 325-1000mg q4-6h, or Naproxen 220-550mg q12h. For bacterial infections, Amoxicillin or Penicillin V. Homeopathic options include Belladonna, Hepar sulph, Merc sol, or Phytolacca. Ayurvedic remedies: Turmeric milk, Licorice, Tulsi tea, Honey/Ginger.\n\nThis is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='27fe90e9-9bec-4057-b757-867cf21cae5c' timestamp=1763451100.270183
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""The provided "Executive Prescription for Throat Inflammation" attempts to consolidate information from different sources but falls short of being a well-written and complete medical prescription for several critical reasons:

1.  **Lack of Specificity for Non-Prescription Medications:** While dosages are provided for Ibuprofen, Acetaminophen, and Naproxen, the prescription doesn't specify whether these are intended for immediate relief or for a defined duration. It also fails to mention maximum daily dosages for OTC use.
2.  **Incomplete Dosage Information for Prescription Medications:** Amoxicillin and Penicillin V are listed, but the dosages are not specific (e.g., "Amoxicillin or Penicillin V" without strength, frequency, or duration, especially considering these are typically prescription-strength for infections). The context from AllopathicAgent provided more detailed dosages, but this aggregated prescription omits them.
3.  **Vague Homeopathic and Ayurvedic Recommendations:** The prescription lists homeopathic and Ayurvedic remedies without any guidance on potency (for homeopathic) or preparation/frequency (for Ayurvedic beyond general ideas). The context from HomeopathicAgent and AyurvedicAgent provided more specific details which are missing here. For example, homeopathic remedies require specific potencies (e.g., 30C, 6X) and frequencies tailored to symptom severity. Ayurvedic remedies also have specific preparations and timings that are not detailed.
4.  **Absence of a Definitive Diagnosis:** A true prescription would stem from a diagnosis. This prescription lists various symptomatic treatments and potential treatments for bacterial infections without establishing whether a bacterial infection is present, which is crucial for antibiotic use.
5.  **No Citations:** The prompt specifically requested citations, which are entirely absent from this prescription.
6.  **Missing Contraindications and Warnings:** For any medication, especially antibiotics and NSAIDs, crucial warnings about contraindications (e.g., allergies, kidney issues for NSAIDs, pregnancy for certain antibiotics) and potential side effects are missing.

**Suggestions for Improvement:**

1.  **Clarify Scope of Practice and Specificity:** For OTC pain relievers, provide recommended maximum daily dosages and a general duration of use (e.g., "for up to 3-5 days"). For prescription antibiotics (Amoxicillin, Penicillin V), include standard adult dosages, frequency, and duration (e.g., "Amoxicillin 500mg twice daily for 7-10 days for suspected strep throat, pending culture results").
2.  **Provide Specific Potencies and Frequencies for Homeopathic Remedies:** If including homeopathic remedies, specify the common potencies and dosing frequencies associated with them (e.g., "Belladonna 30C, 1-2 pellets every 2-3 hours for acute symptoms"). Similarly, for Ayurvedic remedies, include more precise instructions (e.g., "Turmeric milk: 1 teaspoon turmeric in a cup of warm milk, once daily").
3.  **Include Essential Disclaimer and Contextual Information:** While a general disclaimer is present, it could be enhanced to emphasize that this is not a substitute for a physician's diagnosis and that antibiotics should only be used if a bacterial infection is confirmed by a healthcare professional. Add a note about potential interactions between remedies if multiple are being considered."""
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=675,
  prompt_token_count=2107,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2107
    ),
  ],
  total_token_count=2782
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='CriticAgent' actions=EventActions(skip_summarization=None, state_delta={'critique': 'The provided "Executive Prescription for Throat Inflammation" attempts to consolidate information from different sources but falls short of being a well-written and complete medical prescription for several critical reasons:\n\n1.  **Lack of Specificity for Non-Prescription Medications:** While dosages are provided for Ibuprofen, Acetaminophen, and Naproxen, the prescription doesn\'t specify whether these are intended for immediate relief or for a defined duration. It also fails to mention maximum daily dosages for OTC use.\n2.  **Incomplete Dosage Information for Prescription Medications:** Amoxicillin and Penicillin V are listed, but the dosages are not specific (e.g., "Amoxicillin or Penicillin V" without strength, frequency, or duration, especially considering these are typically prescription-strength for infections). The context from AllopathicAgent provided more detailed dosages, but this aggregated prescription omits them.\n3.  **Vague Homeopathic and Ayurvedic Recommendations:** The prescription lists homeopathic and Ayurvedic remedies without any guidance on potency (for homeopathic) or preparation/frequency (for Ayurvedic beyond general ideas). The context from HomeopathicAgent and AyurvedicAgent provided more specific details which are missing here. For example, homeopathic remedies require specific potencies (e.g., 30C, 6X) and frequencies tailored to symptom severity. Ayurvedic remedies also have specific preparations and timings that are not detailed.\n4.  **Absence of a Definitive Diagnosis:** A true prescription would stem from a diagnosis. This prescription lists various symptomatic treatments and potential treatments for bacterial infections without establishing whether a bacterial infection is present, which is crucial for antibiotic use.\n5.  **No Citations:** The prompt specifically requested citations, which are entirely absent from this prescription.\n6.  **Missing Contraindications and Warnings:** For any medication, especially antibiotics and NSAIDs, crucial warnings about contraindications (e.g., allergies, kidney issues for NSAIDs, pregnancy for certain antibiotics) and potential side effects are missing.\n\n**Suggestions for Improvement:**\n\n1.  **Clarify Scope of Practice and Specificity:** For OTC pain relievers, provide recommended maximum daily dosages and a general duration of use (e.g., "for up to 3-5 days"). For prescription antibiotics (Amoxicillin, Penicillin V), include standard adult dosages, frequency, and duration (e.g., "Amoxicillin 500mg twice daily for 7-10 days for suspected strep throat, pending culture results").\n2.  **Provide Specific Potencies and Frequencies for Homeopathic Remedies:** If including homeopathic remedies, specify the common potencies and dosing frequencies associated with them (e.g., "Belladonna 30C, 1-2 pellets every 2-3 hours for acute symptoms"). Similarly, for Ayurvedic remedies, include more precise instructions (e.g., "Turmeric milk: 1 teaspoon turmeric in a cup of warm milk, once daily").\n3.  **Include Essential Disclaimer and Contextual Information:** While a general disclaimer is present, it could be enhanced to emphasize that this is not a substitute for a physician\'s diagnosis and that antibiotics should only be used if a bacterial infection is confirmed by a healthcare professional. Add a note about potential interactions between remedies if multiple are being considered.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='d007597d-4055-4494-afb1-66424fc066af' timestamp=1763451102.051497
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""**Prescription for Throat Inflammation**

**Disclaimer:** This prescription is for informational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment. Always seek the advice of your physician or other qualified health provider with any questions you may have regarding a medical condition. Antibiotics should only be used if a bacterial infection is confirmed by a healthcare professional. AI responses may include errors.

**1. For Pain and Fever (Allopathic - OTC):**

*   **Ibuprofen:** 200-400mg every 4-6 hours as needed.
    *   *Maximum daily OTC dose:* 1200mg in 24 hours.
    *   *Duration:* For up to 3-5 days.
*   **Acetaminophen:** 325-1000mg every 4-6 hours as needed.
    *   *Maximum daily OTC dose:* 4000mg in 24 hours.
    *   *Duration:* For up to 3-5 days.
*   **Naproxen:** 220-550mg every 12 hours.
    *   *Maximum daily OTC dose:* Consult package or healthcare provider.
    *   *Duration:* For up to 3-5 days.

**Important Considerations for OTC Pain Relievers:**
*   Do not exceed recommended dosages.
*   Consult a healthcare provider if you have pre-existing conditions (e.g., kidney problems, allergies) or are taking other medications.
*   Avoid if allergic to NSAIDs or Acetaminophen.

**2. For Suspected Bacterial Infections (Allopathic - Prescription):**

*   **Amoxicillin:**
    *   *Adults (mild to moderate infections):* 500mg every 12 hours or 250mg every 8 hours for 10 days.
    *   *Adults (severe infections):* 875mg every 12 hours or 500mg every 8 hours for 10 days.
    *   *Children:* 50 mg/kg/day in 2 or 3 divided doses for 10 days, or 50 mg/kg once daily (max 1000 mg) for 10 days.
*   **Penicillin V:**
    *   *Adults:* 500mg twice daily or 250mg four times daily for 10 days.
    *   *Children:* 250mg two or three times daily for 10 days (dose adjusted by weight).

**Important Considerations for Antibiotics:**
*   These are typically prescribed for bacterial infections like Strep Throat, pending confirmation (e.g., culture results).
*   Complete the full course of antibiotics as prescribed, even if symptoms improve.
*   Inform your doctor about any allergies (especially to penicillin) or other medications you are taking.
*   Potential side effects can occur; consult your doctor if concerned.

**3. Homeopathic Support:**

*   **Belladonna 30C:** 1-2 pellets every 2-3 hours for acute symptoms with sudden onset, high fever, bright red/swollen throat, and burning pain.
*   **Hepar sulph 30C:** 3 globules every 3 hours for a splinter-like sensation in the throat, worse from cold.
*   **Mercurius solubilis 30C (Merc sol):** 3 globules every 3 hours for raw, burning throat with a feeling of a foreign body, excessive salivation, and bad taste/odor.
*   **Phytolacca decandra 30C:** 3 globules every 3 hours for intense sore throat with a lump/splinter sensation radiating to the ears, dark red/swollen tonsils.

**Important Considerations for Homeopathic Remedies:**
*   Homeopathic treatment is individualized. If symptoms are severe or do not improve, consult a qualified homeopathic practitioner.
*   Product specific instructions (e.g., ThroatCalm tablets) should be followed if using combination products.

**4. Ayurvedic Support:**

*   **Turmeric Milk:** 1 teaspoon turmeric powder in a glass of warm milk, once daily (preferably before bedtime).
*   **Licorice Root (Mulethi/Yashtimadhu):**
    *   1 teaspoon powder with warm water or honey, twice daily.
    *   Chew a small stick or suck on lozenges.
    *   Gargle: Boil 1 teaspoon powder in a cup of warm water.
*   **Tulsi (Holy Basil) Tea:** Boil 10-15 fresh leaves in 2 cups of water until reduced to 1 cup. Drink twice daily.
*   **Honey and Ginger:** Mix 1 teaspoon fresh ginger juice with 1 teaspoon honey. Take 2-3 times daily.

**Important Considerations for Ayurvedic Remedies:**
*   These are generally supportive measures.
*   Specific preparations and timings should be followed for optimal benefit.
*   Consult an Ayurvedic practitioner for personalized recommendations.

**General Advice:**
*   Stay hydrated by drinking plenty of fluids.
*   Rest your voice.
*   Consider gargling with warm salt water (½ teaspoon salt in a glass of warm water) 2-3 times a day.
*   Avoid irritants like smoke.

**Potential Interactions:** While generally considered safe, if you are taking multiple remedies or medications, consult with a healthcare professional about potential interactions."""
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=1184,
  prompt_token_count=3489,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3489
    ),
  ],
  total_token_count=4673
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={'prescription_summary': '**Prescription for Throat Inflammation**\n\n**Disclaimer:** This prescription is for informational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment. Always seek the advice of your physician or other qualified health provider with any questions you may have regarding a medical condition. Antibiotics should only be used if a bacterial infection is confirmed by a healthcare professional. AI responses may include errors.\n\n**1. For Pain and Fever (Allopathic - OTC):**\n\n*   **Ibuprofen:** 200-400mg every 4-6 hours as needed.\n    *   *Maximum daily OTC dose:* 1200mg in 24 hours.\n    *   *Duration:* For up to 3-5 days.\n*   **Acetaminophen:** 325-1000mg every 4-6 hours as needed.\n    *   *Maximum daily OTC dose:* 4000mg in 24 hours.\n    *   *Duration:* For up to 3-5 days.\n*   **Naproxen:** 220-550mg every 12 hours.\n    *   *Maximum daily OTC dose:* Consult package or healthcare provider.\n    *   *Duration:* For up to 3-5 days.\n\n**Important Considerations for OTC Pain Relievers:**\n*   Do not exceed recommended dosages.\n*   Consult a healthcare provider if you have pre-existing conditions (e.g., kidney problems, allergies) or are taking other medications.\n*   Avoid if allergic to NSAIDs or Acetaminophen.\n\n**2. For Suspected Bacterial Infections (Allopathic - Prescription):**\n\n*   **Amoxicillin:**\n    *   *Adults (mild to moderate infections):* 500mg every 12 hours or 250mg every 8 hours for 10 days.\n    *   *Adults (severe infections):* 875mg every 12 hours or 500mg every 8 hours for 10 days.\n    *   *Children:* 50 mg/kg/day in 2 or 3 divided doses for 10 days, or 50 mg/kg once daily (max 1000 mg) for 10 days.\n*   **Penicillin V:**\n    *   *Adults:* 500mg twice daily or 250mg four times daily for 10 days.\n    *   *Children:* 250mg two or three times daily for 10 days (dose adjusted by weight).\n\n**Important Considerations for Antibiotics:**\n*   These are typically prescribed for bacterial infections like Strep Throat, pending confirmation (e.g., culture results).\n*   Complete the full course of antibiotics as prescribed, even if symptoms improve.\n*   Inform your doctor about any allergies (especially to penicillin) or other medications you are taking.\n*   Potential side effects can occur; consult your doctor if concerned.\n\n**3. Homeopathic Support:**\n\n*   **Belladonna 30C:** 1-2 pellets every 2-3 hours for acute symptoms with sudden onset, high fever, bright red/swollen throat, and burning pain.\n*   **Hepar sulph 30C:** 3 globules every 3 hours for a splinter-like sensation in the throat, worse from cold.\n*   **Mercurius solubilis 30C (Merc sol):** 3 globules every 3 hours for raw, burning throat with a feeling of a foreign body, excessive salivation, and bad taste/odor.\n*   **Phytolacca decandra 30C:** 3 globules every 3 hours for intense sore throat with a lump/splinter sensation radiating to the ears, dark red/swollen tonsils.\n\n**Important Considerations for Homeopathic Remedies:**\n*   Homeopathic treatment is individualized. If symptoms are severe or do not improve, consult a qualified homeopathic practitioner.\n*   Product specific instructions (e.g., ThroatCalm tablets) should be followed if using combination products.\n\n**4. Ayurvedic Support:**\n\n*   **Turmeric Milk:** 1 teaspoon turmeric powder in a glass of warm milk, once daily (preferably before bedtime).\n*   **Licorice Root (Mulethi/Yashtimadhu):**\n    *   1 teaspoon powder with warm water or honey, twice daily.\n    *   Chew a small stick or suck on lozenges.\n    *   Gargle: Boil 1 teaspoon powder in a cup of warm water.\n*   **Tulsi (Holy Basil) Tea:** Boil 10-15 fresh leaves in 2 cups of water until reduced to 1 cup. Drink twice daily.\n*   **Honey and Ginger:** Mix 1 teaspoon fresh ginger juice with 1 teaspoon honey. Take 2-3 times daily.\n\n**Important Considerations for Ayurvedic Remedies:**\n*   These are generally supportive measures.\n*   Specific preparations and timings should be followed for optimal benefit.\n*   Consult an Ayurvedic practitioner for personalized recommendations.\n\n**General Advice:**\n*   Stay hydrated by drinking plenty of fluids.\n*   Rest your voice.\n*   Consider gargling with warm salt water (½ teaspoon salt in a glass of warm water) 2-3 times a day.\n*   Avoid irritants like smoke.\n\n**Potential Interactions:** While generally considered safe, if you are taking multiple remedies or medications, consult with a healthcare professional about potential interactions.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='e3269742-a93a-4f5e-a8e3-1e7c1fba59ad' timestamp=1763451106.012617
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text='APPROVED'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=1,
  prompt_token_count=5020,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=5020
    ),
  ],
  total_token_count=5021
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='CriticAgent' actions=EventActions(skip_summarization=None, state_delta={'critique': 'APPROVED'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='cb51910a-7748-4848-87ad-f95be655e3fc' timestamp=1763451111.192904
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={},
        id='adk-c8ed89e3-364c-4644-a826-3dbfbdeaa473',
        name='exit_loop'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=5053,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=5053
    ),
  ],
  total_token_count=5063
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=set() branch=None id='f3d5f0fe-fd32-478c-b6a8-2aa8c917f4fa' timestamp=1763451112.569833
model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='adk-c8ed89e3-364c-4644-a826-3dbfbdeaa473',
        name='exit_loop',
        response={
          'message': 'Prescription approved. Exiting refinement loop.',
          'status': 'approved'
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='aaf8eebd-99f9-4e28-a559-e0460947c83b' timestamp=1763451113.732428
model_version='gemini-2.5-flash-lite' content=Content(
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=5091,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=5091
    ),
  ],
  total_token_count=5091
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-8b0e7f1b-658a-470d-a017-bc2a52784fc5' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='3da165bf-25d4-4ab5-84ff-4f82d620ab8b' timestamp=1763451113.733698